In [ ]:
import os
os.environ["AF3_NB_OVERRIDES"] = "{\"model\": \"openbind0\", \"protein\": \"PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK\", \"msa_mode\": \"single_sequence\", \"num_diffusion_samples\": 1, \"num_recycles\": 3, \"jobname\": \"e2e\"}"
print("overrides:", os.environ["AF3_NB_OVERRIDES"])


overrides: {"model": "openbind0", "protein": "PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK", "msa_mode": "single_sequence", "num_diffusion_samples": 1, "num_recycles": 3, "jobname": "e2e"}


In [ ]:
#@title Install dependencies (~35 s)
# No `%%time`: a cell magic has to be the FIRST line, and `#@title` already
# is -- with both, Colab renders the form and every other runner aborts the
# cell. Timed explicitly below, which also survives being run headlessly.
import os, time, glob, shutil, sys
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]
#@markdown - **model**: which weights to run -- everything downstream is identical.
#@markdown   `openbind0` is a good default; the `esmfold2*` entries fold from ESM-C with
#@markdown   no MSA. Licences, download sizes and notes: the Instructions cell.

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in Drive so the next
#@markdown   session skips the recompile -- worth ~53 s (69 s cold vs 16 s warm on a
#@markdown   68-residue input). Never changes a result.

# HEADLESS OVERRIDES. A form field above is a plain assignment, so a notebook
# run outside Colab -- colab-cli, CI -- has no way to change one. Any field can
# be set from the environment instead:
#     AF3_NB_OVERRIDES='{"model": "boltz2", "msa_mode": "single_sequence"}'
# Unset, this does nothing at all, which is every interactive run.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# PINNED, both halves. Until 2026-09-16 this installed the v3.1.5 wheel for its
# compiled extension and then overlaid the Python half from the BRANCH HEAD --
# so the notebook mixed a fixed binary with a moving source tree, and two runs
# on different days could be different code. `alphafold3-colabfold` is published
# on PyPI (cp312/cp313/cp314 manylinux + macOS arm64) and its Python half knows
# every model, so the overlay is gone and both the package and run_alphafold.py
# come from one tag.
VERSION = '3.1.9'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
IS_AF3 = (model == 'alphafold3')
# AlphaFold 2 is a SIBLING NETWORK, not one of the AF3-family ports: MSA row and
# column attention into an IPA head, reached through the same CLI and writing the
# same outputs. Its parameters are DeepMind's own release under CC BY 4.0, so they
# are fetched from source. Protein only -- a ligand or nucleotide in the input
# raises rather than folding the protein part and saying nothing.
IS_AF2 = model.startswith('af2_')
AF2_DIR = 'af2_params'
# int8 everywhere: same weights stored 8-bit and expanded on load, which is
# what keeps a Colab download to a few hundred MB. Not a knob -- there is no
# reason to pick anything else here, and AlphaFold 3's own parameters come
# from Google as float32 regardless.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # THE SLIM WHEEL, from PyPI. It is 9 MB. Until v3.1.8 this had to be the
  # 130 MB `+data` wheel from a GitHub release, because importing
  # `alphafold3.cpp` died with
  #     ImportError: Could not find the libcifpp components.cif file.
  # unless the 518 MB dictionary was bundled. That error was OURS, not
  # libcifpp's: mkdssp_pybind.cc threw from module REGISTRATION, so the whole
  # extension -- including `cif_dict`, which featurisation needs -- failed to
  # import for the sake of DSSP, which no fold calls. v3.1.8 defers the check
  # to `get_dssp` itself. VERIFIED on the published 3.1.8 wheel in a clean
  # venv with no components.cif anywhere: the extension imports, a
  # protein+ligand input featurises, and get_dssp raises something actionable.
  # ml_collections and dm-tree ARE declared dependencies of the package, but it
  # goes in with --no-deps (so pip does not re-resolve jax and the CUDA stack
  # Colab already has), so every third-party import has to be listed here. Those two are imported ONLY
  # by the af2 path (`af2/model/config.py`, and dm-tree in three more), which is
  # why every af3-family model worked and `--model af2_ptm` died with
  # `ModuleNotFoundError: No module named 'ml_collections'`.
  # The full set under src/alphafold3/af2 is: absl, haiku, jax, ml_collections,
  # numpy, scipy, tree -- the rest are already here or in Colab's base image.
  # MEASURED against a fresh Colab image (2026-09-16, py 3.13.15, T4), not
  # guessed. Already present, so not installed: zstandard 0.25.0, dm-tree
  # 0.1.10, numpy 2.1.3, scipy 1.16.3, absl-py, and 21 nvidia CUDA wheels.
  # Absent, so installed: dm-haiku, rdkit, tokamax, ml_collections, py2Dmol.
  #
  # DROPPED: awscli and py3Dmol were installed and never used -- `aws` is never
  # invoked and only py2Dmol is imported. awscli alone drags in the boto stack.
  # NO JAX PIN. This used to force jax[cuda12]==0.10.1; Colab ships 0.11.1, so
  # the pin downgraded jax AND re-pulled the whole CUDA wheel stack -- the
  # dominant cost of this cell. Measured on a fresh T4 session: these four
  # install in 8.5 s against minutes with the pin, and jax 0.11.1 imports,
  # traces and folds correctly (openbind0, 20 residues, rc=0, 165 atoms).
  #
  # CAVEAT, stated because it is untested rather than dismissed: that check was
  # on a T4, which takes the XLA attention path. tokamax's Triton kernels are
  # restricted to datacenter GPUs below, so an A100/H100 run exercises code a
  # T4 does not. If a datacenter GPU misbehaves, pin jax again here first.
  os.system("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 \
  tokamax==0.0.11 ml_collections")
  # py2Dmol from source: the released wheel lags the repo.
  os.system("pip install -q git+https://github.com/sokrypton/py2Dmol.git")
  # aria2c, for AF2 only: its parameter tar is 5.3 GB and a single connection is
  # the bottleneck, not the link (weights._download_parallel uses it when it is
  # on PATH). Every af3-family blob is 130-350 MB, where this would not pay.
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")
  # --no-deps: the package declares jax, and Colab already ships a working
  # one -- resolving its dependencies would re-pull the whole CUDA wheel
  # stack, which is why every third-party import is listed above instead.
  os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}')
  # `run_alphafold.py` is a top-level script, not part of the package
  # (`wheel.packages = ["src/alphafold3"]`), so the wheel does not carry it.
  # Fetch it AT THE TAG so the driver and the library are the same commit.
  os.system(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
            f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py')
  # haiku 0.0.17 still calls the moved `jax.core.DropVar`; checked against the
  # installed 0.0.17 tree, this one is still needed. (A second sed for
  # `jax.core.get_opaque_trace_state` used to sit here and never matched --
  # base.py reaches it through a `jax_core` alias and already falls back to
  # `jex_core` itself, so it was only ever a no-op.)
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, in the background. The ported models are fetched by the same code the run
# uses (alphafold3.model.weights.ensure_weights), so the run finds them already there
# and the cache layout cannot drift between the two. AlphaFold 3's own parameters are
# not ours to redistribute, so those come straight from Google.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if not os.path.isfile(STAMP):
  if IS_AF2:
    print('Downloading official AlphaFold 2 parameters (CC BY 4.0)...')
    with open('prefetch_af2.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_af2_params(sys.argv[1]))\n')
    os.system(f'(python prefetch_af2.py {AF2_DIR} > {STAMP}.log 2>&1 && touch {STAMP}) &')
  elif IS_AF3:
    print("Downloading official AlphaFold 3 weights (public, no login required)...")
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}" > {STAMP}.log 2>&1 && touch {STAMP}) &')
  else:
    print(f'Downloading {model} weights...')
    with open('prefetch_weights.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n')
    os.system(f'(python prefetch_weights.py {model} {PRECISION} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# Where the compiled model is cached. /tmp is wiped when the VM goes away, so a
# fresh session recompiles (~53 s on a small input); Drive survives. Opt-in, and
# the run falls back to /tmp if the mount does not work rather than failing.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Fetching the CCD components this fold needs...')
  # NOT build_data. That parses libcifpp's whole components.cif -- 51,275
  # components, 518 MB -- into a 505 MB ccd.pickle, and costs 48 s of the
  # session. A fold references about thirty codes.
  #
  # LocalFold's trick: fetch each component from
  # files.rcsb.org/ligands/download/<CODE>.cif, kilobytes each, and build the
  # two pickles from just those. Measured: 0.6 s for 40 components, a 0.30 MB
  # pickle, and every field byte-identical to libcifpp's for ALA, SER, GOL,
  # ATP, SEP, NAG, DA and U.
  #
  # Both pickles come from the SAME fetched set, so they are self-consistent --
  # NAG lands in GLYCAN_LINKING_LIGANDS exactly as with the full dictionary. A
  # component the input names and we did not fetch raises KeyError, which is
  # loud, rather than being silently mis-bonded; hence the generous code list.
  # `ccd_fetch` ships in the wheel as of v3.1.8. It used to be fetched from
  # `main`, which meant the notebook mixed a tagged package with a moving
  # file -- the exact drift the pinned install above exists to prevent.
  with open('prefetch_ccd.py', 'w') as fh:
    fh.write(
      'import sys, os, importlib.metadata as md\n'
      'from alphafold3.constants import ccd_fetch\n'
      'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
      '.locate_file("alphafold3"))\n'
      'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
      'os.makedirs(conv, exist_ok=True)\n'
      'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
      '  os.path.join(conv, "ccd.pickle"),\n'
      '  os.path.join(conv, "chemical_component_sets.pickle"))\n')
  os.system('(python prefetch_ccd.py > DATA_DONE.log 2>&1 && touch DATA_DONE) &')

# A BOUNDED wait. This used to be `while not exists: sleep(5)` with no limit
# and the background job's output discarded, so a failed download was an
# indefinite hang with nothing on screen -- which is exactly how it looked for
# ten minutes on 2026-09-17. Each job now writes <sentinel>.log, and a stall
# raises with the tail of it rather than waiting for the runtime to be
# reclaimed. The weights are the slow one: a few hundred MB, so minutes on a
# poor link, hence 20 of them before giving up.
def _await(sentinel, limit=1200):
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} ✓  ({time.time() - t0:.0f} s)')

for sentinel in (STAMP, 'DATA_DONE'):
  _await(sentinel)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


override: model = 'openbind0'
Installing packages...


Packages installed.


Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).
Fetching the CCD components this fold needs...


WEIGHTS_DONE_openbind0_int8 ✓  (10 s)
DATA_DONE ✓  (0 s)
Setup complete!  Model: openbind0.
Setup took 34 s.


In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# HEADLESS OVERRIDES. A form field above is a plain assignment, so a notebook
# run outside Colab -- colab-cli, CI -- has no way to change one. Any field can
# be set from the environment instead:
#     AF3_NB_OVERRIDES='{"model": "boltz2", "msa_mode": "single_sequence"}'
# Unset, this does nothing at all, which is every interactive run.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


override: model = 'openbind0'
override: protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK'
override: msa_mode = 'single_sequence'
override: jobname = 'e2e'
Job "e2e_09338"  ->  results will be written to af3_output/e2e_09338/


{'name': 'e2e_09338',
 'sequences': [{'protein': {'id': 'A',
    'sequence': 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK',
    'templates': [],
    'unpairedMsa': '>query\nPIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK\n',
    'pairedMsa': ''}}],
 'modelSeeds': [1],
 'dialect': 'alphafold3',
 'version': 1}

In [ ]:
#@title Run the model
# No `%%time` -- see the install cell.
import os, shutil, subprocess, glob, time
_T0 = time.time()

#@markdown Defaults match AlphaFold 3; raise only if needed.
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes; more helps hard targets, costs time.
#@markdown - `num_diffusion_samples`: structures per seed, so total = seeds x samples.

# HEADLESS OVERRIDES. A form field above is a plain assignment, so a notebook
# run outside Colab -- colab-cli, CI -- has no way to change one. Any field can
# be set from the environment instead:
#     AF3_NB_OVERRIDES='{"model": "boltz2", "msa_mode": "single_sequence"}'
# Unset, this does nothing at all, which is every interactive run.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights. Every ported model resolves its own cache directory (populated by the
# install cell), so --model_dir is passed only for the two whose parameters come
# from DeepMind directly: AlphaFold 3's, and AlphaFold 2's (CC BY 4.0, fetched
# into af2_params by the install cell).
print(f'Model: {model}')

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model={model}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    f'--cache_dir={CACHE_DIR}',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
# chai-1 folds from ESM2 and ESMFold2 from ESM-C; without it they are a
# different model, not a slightly worse one (a natural protein goes to 5.70 A
# where chai-1 reaches 0.642, and an ESMFold2 variant with no MSA encoder has
# nothing left to fold from at all). Both towers run in-process and download on
# demand, which is why run_alphafold makes it opt-in and this passes it.
if model == 'chai1' or model.startswith('esmfold2'):
  cmd.append('--use_esm_embeddings')
if nojit:
  cmd.append('--nojit')
if IS_AF3 or IS_AF2:
  cmd.append(f'--model_dir={AF2_DIR if IS_AF2 else NATIVE_DIR}')

cmd = ' '.join(cmd)
if run_it:
  print(cmd)
  # NOT `!{cmd}`. That reports nothing about how the run ended, and the line
  # below used to print `Done -> ...` whatever happened -- so a hard failure
  # (an ImportError, 230 ms) read as a successful fold with no structures.
  # Popen streams the same output AND yields a status.
  _p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
  for _line in _p.stdout:
    print(_line, end='')
  _rc = _p.wait()
  _cifs = glob.glob(f'{job_dir}/**/*.cif', recursive=True)
  if _rc != 0 or not _cifs:
    raise RuntimeError(
        f'the fold FAILED (exit {_rc}, {len(_cifs)} structures written). '
        'The output above is the whole story; scroll up for the error.')
  print(f'\nDone -> {job_dir}/  ({len(_cifs)} structures, '
        f'{time.time() - _T0:.0f} s)')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


override: model = 'openbind0'
override: protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK'
override: msa_mode = 'single_sequence'
override: num_diffusion_samples = 1
override: num_recycles = 3
override: jobname = 'e2e'
Pre-Ampere GPU (compute capability 7.5) - XLA attention + custom-kernel fusion disabled.
XLA_FLAGS = --xla_disable_hlo_passes=custom-kernel-fusion-rewriter
Model: openbind0
python run_alphafold.py --json_path=/tmp/af3_inputs/e2e_09338.json --model=openbind0 --norun_data_pipeline --output_dir=af3_output --cache_dir=/tmp/af3_cache --force_output_dir --flash_attention_implementation=xla --num_recycles=3 --num_diffusion_samples=1


/content/run_alphafold.py:718: DeprecationWarning: backend and device argument on jit is deprecated. You can use `jax.device_put(..., jax.local_devices(backend="cpu")[0])` on the inputs to the jitted function to get the same behavior.
  apply_fn = jax.jit(apply_fn, device=self._device)


W0917 01:11:41.205270 132936496715456 confidences.py:133] e2e_09338: rasa calculation failed: get_dssp needs libcifpp's components.cif, which was not found. Set LIBCIFPP_DATA_DIR to a directory containing it (the wwPDB CCD, ~518 MB). Only DSSP-derived outputs need it -- folding does not.
Found local GPU devices: [CudaDevice(id=0)], using device 0: cuda:0
Building model from scratch...
Checking that model parameters can be loaded...

Running fold job e2e_09338...
Output will be written in af3_output/e2e_09338
Skipping data pipeline...
Writing model input JSON to af3_output/e2e_09338/e2e_09338_data.json
Predicting 3D structure for e2e_09338 with 1 seed(s)...
Featurising data with 1 seed(s)...
Featurising data with seed 1.
Featurising data with seed 1 took 1.29 seconds.
Featurising data with 1 seed(s) took 1.32 seconds.
Running model inference and extracting output structure samples with 1 seed(s)...
Running model inference with seed 1...
Running model inference with seed 1 took 54.62 sec


Done -> af3_output/e2e_09338/  (2 structures, 68 s)


In [ ]:

import glob, json, os
cifs = sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True))
atoms = sum(1 for l in open(cifs[0]) if l.startswith('ATOM')) if cifs else 0
print('CIFS:', len(cifs), 'ATOMS:', atoms)
conf = sorted(glob.glob(f'{job_dir}/**/*summary_confidences.json', recursive=True))
if conf:
  print('PLDDT/PTM:', {k: v for k, v in json.load(open(conf[0])).items()
                       if k in ('ptm', 'iptm', 'fraction_disordered')})
print('RESULT:', 'PASS' if (cifs and atoms > 100) else 'FAIL')


CIFS: 2 ATOMS: 448
PLDDT/PTM: {'fraction_disordered': 0.0, 'iptm': None, 'ptm': 0.58}
RESULT: PASS
